In [8]:
import sys
from pathlib import Path
import geopandas as gpd
from datetime import datetime, timezone
import pandas as pd
from tqdm import tqdm
import shutil

# Find the repository root.
PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, photo_paths
from src.prep import (
    download_photo,
    get_image_metadata,
    calculate_ground_extent,
    ground_to_mercator_extent,
    update_tracker,
)

cfg = load_config()

In [9]:
INDEX_PATH = cfg["paths"]["index"]
TRACKER_PATH = cfg["paths"]["tracker"]
LAYER = "photos"

print("Source index:", INDEX_PATH)
print("Batch tracker:", TRACKER_PATH)
print("County:", cfg["county"])
print("Years:", cfg["years"])
print("Scale:", cfg["scale"])

Source index: C:\Users\eu01251115\LocalProjects\batch_georef\data\main_index_blank.gpkg
Batch tracker: C:\Users\eu01251115\LocalProjects\batch_georef\data\batches\carver_1963_20000_tracker.gpkg
County: Carver
Years: [1963]
Scale: 20000


In [ ]:
# Load the main index.
index_gdf = gpd.read_file(INDEX_PATH, layer=LAYER)

In [ ]:
# examine all years in the county

county_gdf = index_gdf.loc[index_gdf["county"] == cfg["county"]]

for year in sorted(county_gdf["year"].dropna().unique()):
    print(year)

In [ ]:
# Select records for the configured county and years.
mask = (
    index_gdf["county"].str.casefold().eq(
        cfg["county"].casefold()
    )
    & index_gdf["year"].isin(cfg["years"])
)

In [ ]:
batch_gdf = index_gdf.loc[mask].copy()

### Manually determining ground scale:

- Visit the [MHAPO Air Photo Holdings page](https://www.lib.umn.edu/collections/borchert/aerial-photography-and-remote-sensed-imagery). 
- Select your chosen county to open a PDF with metadata. 
- Use this to determine the ground scale for your target year or years. Currently, this tool supports processing multiple years at the same time, but only if they share the same ground scale. 
- Set the scale for your currently batch in config.yml.
- Specify a batch name for the tracker table. The photos and the folders they are in will be named for the county and year. 

#### Once you have configured your year(s) and scale in config.yml, restart the kernel and start again from the beginning.


## Create a tracker table

In [ ]:
if batch_gdf.empty:
    raise ValueError("No photographs match this batch configuration.")

if (
    batch_gdf["cty_year_id"].isna().any()
    or not batch_gdf["cty_year_id"].is_unique
):
    raise ValueError("Batch contains missing or duplicate photo IDs.")

In [ ]:
# check length and confirm CRS
print(f"Selected {len(batch_gdf)} photographs")
print("CRS:", batch_gdf.crs)

In [ ]:
# Assign the configured scale to all selected photographs.
batch_gdf["scale"] = cfg["scale"]

In [ ]:
# preview it
display(
    batch_gdf[
        ["cty_year_id", "county", "year", "scale", "url"]
    ].head()
)

In [ ]:
batch_gdf.columns

In [ ]:
if TRACKER_PATH.exists():
    raise FileExistsError(
        f"Batch tracker already exists: {TRACKER_PATH}"
    )

TRACKER_PATH.parent.mkdir(parents=True, exist_ok=True)

batch_gdf.to_file(
    TRACKER_PATH,
    layer=LAYER,
    driver="GPKG",
)

print(f"Created batch tracker: {TRACKER_PATH}")

## Download JPEGS and calculate extents

In [ ]:
# Load the batch tracker for preparation
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

print(f"Batch: {cfg['batch_name']}")
print(f"Photographs: {len(tracker_gdf)}")

display(tracker_gdf.head())

In [ ]:
# Download original JPEGs.
download_results = []

for _, photo in tqdm(
    tracker_gdf.iterrows(),
    total=len(tracker_gdf),
    desc="Downloading photographs",
):
    photo_id = str(photo["cty_year_id"])
    jpeg_path = photo_paths(photo, cfg)["jpeg"]

    try:
        download_photo(photo["url"], jpeg_path)

        download_results.append({
            "cty_year_id": photo_id,
            "jpeg_path": str(jpeg_path),
            "error": None,
        })

    except Exception as exc:
        print(f"FAILED: {photo_id}: {exc}")

        download_results.append({
            "cty_year_id": photo_id,
            "jpeg_path": str(jpeg_path),
            "error": str(exc),
        })

download_df = pd.DataFrame(download_results)

print(f"Successful downloads: {download_df['error'].isna().sum()}")
print(f"Failed downloads: {download_df['error'].notna().sum()}")

display(download_df.head())

In [ ]:
# Collect preparation results for each photo.
results = []

for _, photo in tqdm(
    tracker_gdf.iterrows(),
    total=len(tracker_gdf),
    desc="Calculating extents",
):
    photo_id = photo["cty_year_id"]
    image_path = photo_paths(photo, cfg)["jpeg"]

    result = {
        "cty_year_id": photo_id,
        "width_px": None,
        "height_px": None,
        "dpi_x": None,
        "dpi_y": None,
        "ground_width": None,
        "ground_height": None,
        "wm_width": None,
        "wm_height": None,
        "date_prepared": None,
        "error": None,
    }

    try:
        # Read metadata from the existing JPEG.
        metadata = get_image_metadata(image_path)
        result.update(metadata)

        # Calculate ground dimensions in meters.
        result["ground_width"] = calculate_ground_extent(
            pixels=metadata["width_px"],
            dpi=metadata["dpi_x"],
            scale=photo["scale"],
        )

        result["ground_height"] = calculate_ground_extent(
            pixels=metadata["height_px"],
            dpi=metadata["dpi_y"],
            scale=photo["scale"],
        )

        # Convert dimensions to Web Mercator.
        result["wm_width"] = ground_to_mercator_extent(
            ground_extent_m=result["ground_width"],
            latitude=photo["latitude"],
        )

        result["wm_height"] = ground_to_mercator_extent(
            ground_extent_m=result["ground_height"],
            latitude=photo["latitude"],
        )

        # Mark preparation complete only after successful calculations.
        result["date_prepared"] = datetime.now(
            timezone.utc
        ).date()

    except Exception as e:
        result["error"] = str(e)
        print(f"FAILED: {photo_id}: {e}")

    results.append(result)

# Convert results to a DataFrame for inspection.
results_df = pd.DataFrame(results)

print(f"Successful: {results_df['error'].isna().sum()}")
print(f"Failed: {results_df['error'].notna().sum()}")

display(results_df.head())

In [ ]:
# Update the tracker table
updated_count = update_tracker(
    TRACKER_PATH,
    results_df,
    layer=LAYER,
)

print(f"Updated {updated_count} records.")

### Final load & preview

In [ ]:
tracker_gdf = gpd.read_file(TRACKER_PATH, layer=LAYER)

tracker_gdf